# Incremental Fine-Tuning via MCP

An MCP-native, incremental LoRA/QLoRA fine-tuning experiment for an LLM
**tool-using agent** that must recreate or beat a target paper's MCP results.

**Contract:** the notebook is the control plane; every operation below is an
MCP tool call returning structured JSON. Training/eval/data logic lives in the
Python modules under `src/`.

## 1. Connect to the local MCP servers

This cell starts the four MCP servers (dataset / train / eval / experiment)
over stdio and lists every available tool. That is the "first notebook cell
that lists available tools".


In [ ]:
import asyncio
from mcp_client import Client

mcp = Client(project_dir="..")          # point at the experiment directory
await mcp.connect()

tools = await mcp.list_tools()
print(f"{len(tools)} MCP tools available:\n")
for t in tools:
    print(" -", t["name"])
    if t["description"]:
        print("     ", t["description"])

## 2. Create the experiment

Tell the scaffold the project name, base model and the paper. `paper.json`
holds the authors' reported numbers used by `compare_to_paper` — fill in
`reported_metrics` for your target paper.


In [ ]:
paper = {
  "title": "Your target paper",
  "arxiv_id": "XXXX.XXXXX",
  "reported_metrics": {            # fill these from the paper
    "accuracy": 0.78,
    "success_rate": 0.71
  }
}

await mcp.call("mcp.experiment.create", {
    "name": "mcp-agent-finetune",
    "base_model": "Qwen/Qwen2.5-0.5B-Instruct",   # or your preferred base
    "paper": paper,
    "seed": 0,
})

## 3. Generate + split data

Deterministic synthetic trajectories / teacher tool-calls / rubric labels,
then split into train / val / test. Re-running with the same seed reproduces
the same files (content-hashed).


In [ ]:
await mcp.call("mcp.dataset.generate", {
    "name": "mcp_tool_use", "n_trajectories": 200,
    "n_teacher": 200, "n_rubric": 100, "seed": 0,
})
splits = await mcp.call("mcp.dataset.split", {"name": "mcp_tool_use", "seed": 0})
print({k: v["records"] for k, v in splits.items()})
await mcp.call("mcp.dataset.validate", {"name": "mcp_tool_use_train"})

## 4. Stage 0 — base + initial SFT

Create `stage_0` and run the first fine-tuning stage from the base model.
*(Training runs in deterministic simulate mode here; install
`requirements-train.txt` and set `training.real=true` for real LoRA.)*


In [ ]:
await mcp.call("mcp.experiment.update_config", {
    "updates": {"training": {"method": "lora", "lora_rank": 8, "epochs": 1,
                             "learning_rate": 5e-5}}
})
await mcp.call("mcp.experiment.create_stage", {
    "stage_id": "stage_0", "name": "Base / initial SFT",
    "data_hashes": [splits["mcp_tool_use_train"]["sha256"]],
})
r0 = await mcp.call("mcp.train.start_stage", {
    "stage_id": "stage_0", "new_data": "mcp_tool_use_train", "epochs": 1, "lr": 5e-5, "lora_rank": 8,
})
print(r0["adapter"]["metrics"])

## 5. Evaluate + compare to the paper

Every stage is judged against the authors' reported numbers automatically.


In [ ]:
ev = await mcp.call("mcp.eval.run", {"stage_id": "stage_0", "subset": 200})
print("eval:", ev["metrics"])
cmp0 = await mcp.call("mcp.eval.compare_to_paper", {"stage_id": "stage_0"})
print("vs paper:", cmp0["table"])

## 6. Stage N — incremental fine-tuning

Load the previous adapter, add new data, change hyperparameters, train a short
stage, evaluate, then decide the next stage.


In [ ]:
await mcp.call("mcp.dataset.add_incremental", {
    "name": "mcp_tool_use_incremental",
    "records": [
        {"kind": "trajectory",
         "messages": [{"role": "user", "content": "Use the finance MCP to price a bond."},
                      {"role": "assistant", "content": "Calling mcp.finance.price..."}],
         "tool_calls": [{"name": "mcp.finance.price", "arguments": {"isin": "US0378331005"}}],
         "expected": "mcp.finance.price"}
    ],
})
await mcp.call("mcp.train.set_hyperparams", {
    "updates": {"learning_rate": 2e-5, "epochs": 1, "lora_rank": 16, "method": "lora"}
})
await mcp.call("mcp.experiment.create_stage", {
    "stage_id": "stage_1", "name": "Incremental (new finance data)", "parent": "stage_0"
})
r1 = await mcp.call("mcp.train.start_stage", {
    "stage_id": "stage_1", "from_adapter": "stage_0-adapter",
    "new_data": "mcp_tool_use_incremental", "lora_rank": 16,
})
print(r1["adapter"]["metrics"])
await mcp.call("mcp.eval.compare_to_paper", {"stage_id": "stage_1"})

## 7. Failure cases, judge, report, rollback

Inspect where the model still fails, optionally LLM-judge outputs, export a
lab-notebook report, and show how to roll back a stage.


In [ ]:
fc = await mcp.call("mcp.eval.failure_cases", {"stage_id": "stage_1", "top_k": 5})
print("failures:", fc["count"])

judge = await mcp.call("mcp.eval.llm_judge", {"stage_id": "stage_1", "samples": 10})
print("judge:", judge.get("warning") or judge.get("judge_accuracy"))

rep = await mcp.call("mcp.experiment.export_report", {"stage_id": "stage_1"})
print("report:", rep["report_path"])

# Branching / rollback: drop anything created after stage_0 and start again.
# rb = await mcp.call("mcp.experiment.rollback_to_stage", {"stage_id": "stage_0"})
# print("removed:", rb["removed_stages"])

## 8. Cleanup

Close the MCP connections.


In [ ]:
await mcp.close()